In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [6]:
df = pd.read_csv("Toxicity.csv")
df. head()
x = df.drop("Class", axis =1)
y=df["Class"]
(x.var() == 0).sum()
x.var().sort_values().head(10)

ETA_dAlpha_A    6.570760e-07
AATSC4c         2.031838e-06
AATSC3c         2.155254e-06
AATSC2c         2.379464e-06
AATSC5c         3.050667e-06
AVP-7           3.117846e-06
AATSC7c         3.292758e-06
JGI10           3.295571e-06
AATSC6c         3.381384e-06
JGI9            3.909966e-06
dtype: float64

In [7]:
from sklearn.feature_selection import VarianceThreshold
constant_filter = VarianceThreshold(threshold=0)
x_constant_removed = constant_filter.fit_transform(x)
x_constant_removed.shape

(171, 1203)

In [8]:
low_variance_filter = VarianceThreshold(threshold=0.00001)
x_low_var = low_variance_filter.fit_transform(x)
x_low_var.shape

(171, 1189)

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_encoded[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_low_var)

In [11]:
x_low_var_df = pd.DataFrame(x_low_var)
corr_matrix = x_low_var_df.corr().abs()
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.9)]

len(to_drop)

x_uncorr = x_low_var_df.drop(columns=to_drop)

x_uncorr.shape

(171, 472)

In [12]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_uncorr,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

In [2]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

In [14]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    rf,
    x_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print("F1 scores per fold:", cv_scores)
print("Mean F1:", cv_scores.mean())

F1 scores per fold: [0.         0.28571429 0.16666667 0.18181818 0.36363636]
Mean F1: 0.19956709956709956


In [15]:
rf.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [16]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf.predict(x_test)
y_prob = rf.predict_proba(x_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.70      0.96      0.81        24
           1       0.50      0.09      0.15        11

    accuracy                           0.69        35
   macro avg       0.60      0.52      0.48        35
weighted avg       0.64      0.69      0.60        35

ROC-AUC: 0.5946969696969697


In [17]:
importances = rf.feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": x_uncorr.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feature_importance_df.head(20)

,feature,importance
209,399,0.011200
69,97,0.009173
70,98,0.008520
422,981,0.008442
439,1061,0.007814
123,196,0.007806
271,541,0.007778
86,129,0.007720
159,269,0.007563
131,216,0.007313


In [18]:
(feature_importance_df["importance"] < 0.001).sum()

121

In [21]:
top_k = 50

top_features = feature_importance_df.head(top_k)["feature"]

x_reduced = x_uncorr[top_features]
x_reduced.shape

(171, 50)

In [22]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_reduced,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

In [23]:
from sklearn.ensemble import RandomForestClassifier

rf_reduced = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf_reduced.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [24]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    rf_reduced,
    x_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print("Mean CV F1:", cv_scores.mean())

Mean CV F1: 0.29238095238095235


In [26]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf_reduced.predict(x_test)
y_prob = rf_reduced.predict_proba(x_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.69      0.92      0.79        24
           1       0.33      0.09      0.14        11

    accuracy                           0.66        35
   macro avg       0.51      0.50      0.46        35
weighted avg       0.58      0.66      0.58        35

ROC-AUC: 0.6590909090909091


In [27]:
y_prob = rf_reduced.predict_proba(x_test)[:, 1]

y_pred_adjusted = (y_prob >= 0.35).astype(int)

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_adjusted))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.72      0.75      0.73        24
           1       0.40      0.36      0.38        11

    accuracy                           0.63        35
   macro avg       0.56      0.56      0.56        35
weighted avg       0.62      0.63      0.62        35

ROC-AUC: 0.6590909090909091
